# Product Label Prediction (5 Labels) — Merged with Raw Reddit Text
This notebook fixes the `KeyError` by rebuilding the text from `reddit_comments.csv` and `reddit_posts.csv`, then merging it to the pair-level file using `ext_id` ↔ source `id`.

In [ ]:

from pathlib import Path
import os, re, json, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis")
PAIR_PATH = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/data_processing/ data/ final/pair_level_gpt41mini_clean.csv")
COMMENTS_PATH = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/FInal Project/reddit_comments.csv")
POSTS_PATH = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/FInal Project/reddit_posts.csv")
OUTPUT_DIR = PROJECT_ROOT / "Modeling" / "product_label_prediction_outputs_fixed_merge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PAIR exists   :", PAIR_PATH.exists(), PAIR_PATH)
print("COMMENTS exists:", COMMENTS_PATH.exists(), COMMENTS_PATH)
print("POSTS exists   :", POSTS_PATH.exists(), POSTS_PATH)
print("OUTPUT_DIR     :", OUTPUT_DIR)


In [ ]:

import pandas as pd
import numpy as np

pair_df = pd.read_csv(PAIR_PATH)
comments_df = pd.read_csv(COMMENTS_PATH)
posts_df = pd.read_csv(POSTS_PATH)

print("pair_df:", pair_df.shape)
print("comments_df:", comments_df.shape)
print("posts_df:", posts_df.shape)
display(pair_df.head())
display(comments_df.head(2))
display(posts_df.head(2))


## Build a unified raw-text table
- `reddit_comments.csv`: text comes from `body`
- `reddit_posts.csv`: text comes from `title + selftext`
- merge key: `ext_id` in pair-level file to `id` in raw files

In [ ]:

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

comments_src = comments_df.copy()
comments_src["raw_text"] = comments_src["body"].map(safe_text)
comments_src["source"] = "comment"

posts_src = posts_df.copy()
posts_src["raw_text"] = (
    posts_src["title"].map(safe_text).str.strip() + " " +
    posts_src["selftext"].map(safe_text).str.strip()
).str.strip()
posts_src["source"] = "post"

common_cols = ["id", "raw_text", "source", "subreddit", "created_utc", "searched_keyword"]
for col in common_cols:
    if col not in comments_src.columns:
        comments_src[col] = np.nan
    if col not in posts_src.columns:
        posts_src[col] = np.nan

raw_src = pd.concat([
    comments_src[common_cols],
    posts_src[common_cols]
], ignore_index=True)

raw_src["id"] = raw_src["id"].astype(str)
pair_df["ext_id"] = pair_df["ext_id"].astype(str)

print(raw_src.shape)
display(raw_src.head())


In [ ]:

merged = pair_df.merge(
    raw_src,
    left_on="ext_id",
    right_on="id",
    how="left",
    validate="m:1"
)

print("merged shape:", merged.shape)
print("Matched text rows:", merged["raw_text"].notna().sum(), "of", len(merged))
print("Missing text rows:", merged["raw_text"].isna().sum())
display(merged.head())


In [ ]:

# if a few rows fail to match because of formatting, try trimmed matching
if merged["raw_text"].isna().sum() > 0:
    pair_tmp = pair_df.copy()
    raw_tmp = raw_src.copy()
    pair_tmp["ext_id_norm"] = pair_tmp["ext_id"].astype(str).str.strip()
    raw_tmp["id_norm"] = raw_tmp["id"].astype(str).str.strip()
    merged = pair_tmp.merge(
        raw_tmp.drop(columns=["id"]).rename(columns={"id_norm":"id_norm_raw"}),
        left_on="ext_id_norm",
        right_on="id_norm_raw",
        how="left"
    )
    print("After normalized merge, missing text rows:", merged["raw_text"].isna().sum())


## Keep the 5 product labels and pivot to text-level multi-label data

In [ ]:

ALLOWED_PRODUCTS = ["flower", "oil", "gummies", "vape", "topical"]

df = merged.copy()
df["product"] = df["product"].astype(str).str.lower().str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["product"].isin(ALLOWED_PRODUCTS)].copy()
df = df[df["raw_text"].notna() & (df["raw_text"].astype(str).str.strip() != "")].copy()

print("Filtered pair rows:", df.shape)
print(df["product"].value_counts())


In [ ]:

# Build one row per ext_id / text
text_meta = (
    df.sort_values(["ext_id"])
      .drop_duplicates("ext_id")
      [["ext_id", "raw_text", "source", "subreddit", "created_utc", "searched_keyword"]]
      .rename(columns={"ext_id":"text_id", "raw_text":"text"})
      .reset_index(drop=True)
)

indicator = (
    pd.crosstab(df["ext_id"], df["product"])
      .reindex(columns=ALLOWED_PRODUCTS, fill_value=0)
      .clip(upper=1)
      .reset_index()
      .rename(columns={"ext_id":"text_id"})
)

text_level = text_meta.merge(indicator, on="text_id", how="inner")
for c in ALLOWED_PRODUCTS:
    text_level[c] = text_level[c].astype(int)

text_level["n_labels"] = text_level[ALLOWED_PRODUCTS].sum(axis=1)
text_level["is_multilabel"] = (text_level["n_labels"] > 1).astype(int)

print("text_level:", text_level.shape)
display(text_level.head())
print(text_level[ALLOWED_PRODUCTS].sum())
print("Multilabel rate:", text_level["is_multilabel"].mean())


In [ ]:

# Save dataset
text_level_path = OUTPUT_DIR / "text_level_5label_from_pair_and_raw.csv"
text_level.to_csv(text_level_path, index=False)
print("Saved:", text_level_path)


## Train / validation / test split by `text_id`

In [ ]:

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(text_level, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

X_train = train_df["text"].fillna("").astype(str).tolist()
X_val   = val_df["text"].fillna("").astype(str).tolist()
X_test  = test_df["text"].fillna("").astype(str).tolist()

y_train = train_df[ALLOWED_PRODUCTS].values
y_val   = val_df[ALLOWED_PRODUCTS].values
y_test  = test_df[ALLOWED_PRODUCTS].values

print("train:", len(X_train), y_train.shape)
print("val  :", len(X_val), y_val.shape)
print("test :", len(X_test), y_test.shape)


## Alternative vectorization comparison
This version avoids TF-IDF and compares:
- CountVectorizer
- HashingVectorizer
- Character n-grams
- Word2Vec average embeddings

In [ ]:

from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
import numpy as np
import pandas as pd


In [ ]:

def decision_to_unit_interval(scores):
    scores = np.asarray(scores)
    return 1 / (1 + np.exp(-scores))

def predict_scores(model, X):
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        s = np.asarray(s)
        if s.ndim == 1:
            s = s.reshape(-1, 1)
        return s
    elif hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if isinstance(p, list):
            return np.column_stack([pp[:,1] for pp in p])
        p = np.asarray(p)
        if p.ndim == 3:
            return p[:,:,1]
        return p
    else:
        raise ValueError("Model has neither decision_function nor predict_proba")

def tune_thresholds(y_true, score_mat, grid=None):
    if grid is None:
        grid = np.arange(0.10, 0.91, 0.05)
    best = []
    probs = decision_to_unit_interval(score_mat)
    for j in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1
        for t in grid:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best.append(best_t)
    return np.array(best)

def apply_thresholds(score_mat, thresholds):
    probs = decision_to_unit_interval(score_mat)
    return (probs >= thresholds.reshape(1, -1)).astype(int)

def evaluate_multilabel(y_true, y_pred, label_names, model_name):
    res = {
        "model": model_name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }
    per_label = pd.DataFrame({
        "label": label_names,
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=None, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=None, zero_division=0),
    })
    return res, per_label


In [ ]:

# 1) CountVectorizer + LinearSVC
count_model = Pipeline([
    ("vec", CountVectorizer(lowercase=True, strip_accents="unicode",
                            min_df=1, ngram_range=(1,2))),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])
count_model.fit(X_train, y_train)
count_val_scores = predict_scores(count_model, X_val)
count_thresholds = tune_thresholds(y_val, count_val_scores)
count_test_pred = apply_thresholds(predict_scores(count_model, X_test), count_thresholds)
count_res, count_per = evaluate_multilabel(y_test, count_test_pred, ALLOWED_PRODUCTS, "Count + LinearSVC")
count_res, count_thresholds


In [ ]:

# 2) HashingVectorizer + LinearSVC
hash_model = Pipeline([
    ("vec", HashingVectorizer(lowercase=True, strip_accents="unicode",
                              n_features=2**18, alternate_sign=False,
                              ngram_range=(1,2))),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])
hash_model.fit(X_train, y_train)
hash_val_scores = predict_scores(hash_model, X_val)
hash_thresholds = tune_thresholds(y_val, hash_val_scores)
hash_test_pred = apply_thresholds(predict_scores(hash_model, X_test), hash_thresholds)
hash_res, hash_per = evaluate_multilabel(y_test, hash_test_pred, ALLOWED_PRODUCTS, "Hashing + LinearSVC")
hash_res, hash_thresholds


In [ ]:

# 3) Character n-grams + Logistic Regression
char_model = Pipeline([
    ("vec", CountVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
    ("clf", OneVsRestClassifier(LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear")))
])
char_model.fit(X_train, y_train)
char_val_scores = predict_scores(char_model, X_val)
char_thresholds = tune_thresholds(y_val, char_val_scores)
char_test_pred = apply_thresholds(predict_scores(char_model, X_test), char_thresholds)
char_res, char_per = evaluate_multilabel(y_test, char_test_pred, ALLOWED_PRODUCTS, "Char n-grams + LogReg")
char_res, char_thresholds


In [ ]:

# 4) Word2Vec average embeddings + Logistic Regression
from gensim.models import Word2Vec
import re

def tokenize_for_w2v(text):
    return re.findall(r"[A-Za-z0-9_']+", str(text).lower())

X_train_tok = [tokenize_for_w2v(t) for t in X_train]
X_val_tok = [tokenize_for_w2v(t) for t in X_val]
X_test_tok = [tokenize_for_w2v(t) for t in X_test]

w2v = Word2Vec(
    sentences=X_train_tok,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=10
)

def avg_w2v(tokens, model, dim=100):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_w2v = np.vstack([avg_w2v(t, w2v, 100) for t in X_train_tok])
X_val_w2v   = np.vstack([avg_w2v(t, w2v, 100) for t in X_val_tok])
X_test_w2v  = np.vstack([avg_w2v(t, w2v, 100) for t in X_test_tok])

w2v_model = OneVsRestClassifier(
    LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear")
)
w2v_model.fit(X_train_w2v, y_train)
w2v_val_scores = predict_scores(w2v_model, X_val_w2v)
w2v_thresholds = tune_thresholds(y_val, w2v_val_scores)
w2v_test_pred = apply_thresholds(predict_scores(w2v_model, X_test_w2v), w2v_thresholds)
w2v_res, w2v_per = evaluate_multilabel(y_test, w2v_test_pred, ALLOWED_PRODUCTS, "Word2Vec avg + LogReg")
w2v_res, w2v_thresholds


In [ ]:

comparison = pd.DataFrame([count_res, hash_res, char_res, w2v_res]).sort_values(
    ["macro_f1", "micro_f1"], ascending=False
).reset_index(drop=True)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "vectorization_model_comparison.csv", index=False)


In [ ]:

per_label_all = pd.concat([
    count_per.assign(model="Count + LinearSVC"),
    hash_per.assign(model="Hashing + LinearSVC"),
    char_per.assign(model="Char n-grams + LogReg"),
    w2v_per.assign(model="Word2Vec avg + LogReg"),
], ignore_index=True)
display(per_label_all)
per_label_all.to_csv(OUTPUT_DIR / "per_label_results.csv", index=False)


## Best model and sample error analysis

In [ ]:

best_model_name = comparison.loc[0, "model"]
print("Best model:", best_model_name)

if best_model_name == "Count + LinearSVC":
    best_pred = count_test_pred
    best_thresholds = count_thresholds
elif best_model_name == "Hashing + LinearSVC":
    best_pred = hash_test_pred
    best_thresholds = hash_thresholds
elif best_model_name == "Char n-grams + LogReg":
    best_pred = char_test_pred
    best_thresholds = char_thresholds
else:
    best_pred = w2v_test_pred
    best_thresholds = w2v_thresholds

print("Best thresholds:", dict(zip(ALLOWED_PRODUCTS, best_thresholds)))


In [ ]:

error_rows = []
for i, (yt, yp) in enumerate(zip(y_test, best_pred)):
    if not np.array_equal(yt, yp):
        error_rows.append({
            "text": X_test[i][:500],
            "true_labels": [ALLOWED_PRODUCTS[j] for j in range(len(ALLOWED_PRODUCTS)) if yt[j] == 1],
            "pred_labels": [ALLOWED_PRODUCTS[j] for j in range(len(ALLOWED_PRODUCTS)) if yp[j] == 1],
        })
errors_df = pd.DataFrame(error_rows)
display(errors_df.head(20))
errors_df.to_csv(OUTPUT_DIR / "error_examples.csv", index=False)
print("Saved error examples.")
